In [ ]:
# BLOCO 1: Instalações que Exigem Reinício
!pip install "numpy<2.0"
!pip install "av==15.0.0"
# Instala pyannote antecipado para evitar conflito de numpy no restart
!pip install "pyannote.audio==3.3.2"

print("\n-------------------------------------------------------------------------")
print("✅ INSTALAÇÕES INICIAIS CONCLUÍDAS.")
print("🔴 AÇÃO CRÍTICA: Reinicie a sessão agora.")
print("-------------------------------------------------------------------------")

In [ ]:
# BLOCO 2: Instalações Finais (Execute SOMENTE APÓS o REINÍCIO)
!pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu118
!pip install ffmpeg-python pandas scipy ctranslate2 faster-whisper soundfile

print("\n✅ INSTALAÇÃO COMPLETA. Ambiente pronto.")

In [ ]:
# BLOCO 3: MONTAGEM DO DRIVE E CONFIGURAÇÃO DE SESSÃO
from google.colab import drive
import os
import sys

drive.mount('/content/drive')

# ----------------------------------------------------------------------
# EDITE APENAS ESTAS VARIÁVEIS PARA CADA NOVA SESSÃO:
# ----------------------------------------------------------------------

ID_MESA = "ID"
NUM_SESSAO = "39"
PARTE_ARQUIVO = ""  # Deixe como "" para arquivo único

ID_SESSAO = f"{ID_MESA}{NUM_SESSAO}"

PASTA_PROJETO = "/content/drive/MyDrive/rpg-transcription"

FORCAR_ATUALIZACAO = False  # Mude para True se quiser puxar mudanças do repo

# ----------------------------------------------------------------------
# CLONAGEM / ATUALIZAÇÃO DO REPOSITÓRIO
# ----------------------------------------------------------------------

if not os.path.exists('rpg_transcription'):
    !git clone https://github.com/ThaisMosken/rpg_transcription.git
elif FORCAR_ATUALIZACAO:
    !git -C rpg_transcription pull

sys.path.append('/content/rpg_transcription')

# ----------------------------------------------------------------------
# DEFINE O GLOSSÁRIO
# ----------------------------------------------------------------------

from utils.glossary_manager import GlossaryManager

manager = GlossaryManager(ID_MESA)
glossario_contexto = manager.get_full_glossary()

GLOSSARIO_NOMES = [
    linha.strip("- *").strip()
    for linha in glossario_contexto.split('\n')
    if linha.strip() and not linha.startswith('#')
]

print(f"✅ Glossário para '{ID_MESA}' carregado com sucesso!")

# ----------------------------------------------------------------------
# VARIÁVEIS DERIVADAS (Não precisam ser editadas)
# ----------------------------------------------------------------------

if PARTE_ARQUIVO:
    ARQUIVO_ENTRADA = os.path.join(PASTA_PROJETO, f"{ID_SESSAO}_parte_{PARTE_ARQUIVO}.wav")
    ARQUIVO_TXT_SAIDA = os.path.join(PASTA_PROJETO, f"{ID_SESSAO}_transcricao_final_parte_{PARTE_ARQUIVO}.txt")
else:
    ARQUIVO_ENTRADA = os.path.join(PASTA_PROJETO, f"{ID_SESSAO}w.wav")
    ARQUIVO_TXT_SAIDA = os.path.join(PASTA_PROJETO, f"{ID_SESSAO}_transcricao_final.txt")

ARQUIVO_CRONICA_SAIDA = os.path.join(PASTA_PROJETO, f"{ID_SESSAO}_cronica.md")

print("Configuração de sessão concluída:")
print(f"  ID da Sessão: {ID_SESSAO}")
print(f"  Entrada WAV: {ARQUIVO_ENTRADA}")
print(f"  Saída TXT: {ARQUIVO_TXT_SAIDA}")
print(f"  Saída Crônica: {ARQUIVO_CRONICA_SAIDA}")

In [ ]:
# BLOCO 4B: Transcrição + Diarização + Alinhamento
from google.colab import userdata
from src.transcription_processor import executar_transcricao_com_segmentos
from src.diarization_processor    import executar_diarizacao
from src.alignment_processor      import alinhar_transcricao_com_diarizacao, salvar_transcricao_diarizada

# Pega o token HF do Secret do Colab
HF_TOKEN = userdata.get('HF_TOKEN')  # salve seu token com este nome nos Secrets

# --- Etapa 1: Transcrição ---
segmentos_whisper = executar_transcricao_com_segmentos(
    arquivo_entrada=ARQUIVO_ENTRADA,
    glossario_nomes=GLOSSARIO_NOMES,
    dispositivo="cuda",
    precisao_modelo="float16",
    nome_modelo="large-v2"
)

# --- Etapa 2: Diarização ---
segmentos_diarizacao = executar_diarizacao(
    arquivo_entrada=ARQUIVO_ENTRADA,
    hf_token=HF_TOKEN
    # num_falantes=5  # descomente e ajuste se souber o número exato
)

# --- Etapa 3: Alinhamento e salvamento ---
segmentos_alinhados = alinhar_transcricao_com_diarizacao(segmentos_whisper, segmentos_diarizacao)
salvar_transcricao_diarizada(segmentos_alinhados, ARQUIVO_TXT_SAIDA)

print("\n🎲 Pipeline completo finalizado!")

In [ ]:
# BLOCO 5: Geração da Crônica
from google.colab import userdata
from src.chronicler import gerar_cronica_gemini

# 1. Tenta pegar a chave do Secret do Colab
try:
    CHAVE_GEMINI = userdata.get('GEMINI_API_KEY')
except:
    CHAVE_GEMINI = None
    print("ERRO: Secret 'GEMINI_API_KEY' não encontrado.")

# 2. Define o modelo
MODELO_ESCOLHIDO = 'gemini-2.5-flash'

# 3. Executa a função importada
if CHAVE_GEMINI:
    gerar_cronica_gemini(
        api_key=CHAVE_GEMINI,
        caminho_transcricao=ARQUIVO_TXT_SAIDA,
        caminho_saida=ARQUIVO_CRONICA_SAIDA,
        glossario_contexto=glossario_contexto,
        modelo=MODELO_ESCOLHIDO
    )